# Test GROMACS

In [38]:
import gmxapi as gmx
import os, shutil
import pathlib


# This just checks if the 'mdrun' resource is available to the API
try:
    # We don't need a real file for this check, just looking at the object
    md = gmx.simulation.mdrun
    print("GPU-enabled mdrun is accessible via gmxapi.")
except AttributeError:
    print("Could not find mdrun module in gmxapi.")

GPU-enabled mdrun is accessible via gmxapi.


## Phase 1 — system preparation Produce a physically valid, force-field-consistent starting configuration

In [12]:
### Step 1 — PDB curation (pure Python, no gmxapi yet)

In [35]:
notebook_dir = pathlib.Path(os.getcwd())          # where your notebook lives
data_dir     = notebook_dir.parent / 'data'       # ../data resolved absolutely
work_dir     = notebook_dir.parent / 'gmx_work'          # output landing directory
work_dir.mkdir(exist_ok=True)

pdb_input = data_dir / 'protein_clean.pdb'

# Sanity check before handing to gmxapi — saves debugging blind
assert pdb_input.exists(), f"PDB not found at: {pdb_input}"
print(f"Input:    {pdb_input}")
print(f"Work dir: {work_dir}")

Input:    /home/biplab/pys/gromacs/data/protein_clean.pdb
Work dir: /home/biplab/pys/gromacs/gmx_work


In [72]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2 — Generate topology with pdb2gmx
# ═══════════════════════════════════════════════════════════════════════════════
# Physical objective:
#   Convert the cleaned PDB into a GROMACS-native coordinate file (.gro) and
#   molecular topology (.top), adding all missing hydrogens and assigning
#   bonded/nonbonded parameters from CHARMM36-jul2022.
#
# Force field choice — CHARMM36-jul2022 + TIP3P:
#   CHARMM36 is the current community standard for membrane and globular
#   proteins. The July 2022 release includes corrected backbone CMAP terms and
#   updated lipid parameters. TIP3P is the water model the FF was optimised
#   against — mixing water models biases hydration free energies.
#
# Key outputs:
#   processed.gro  — all-atom coordinates with GROMACS residue naming
#   topol.top      — full system topology (FF includes + molecule counts)
#   posre.itp      — heavy-atom position restraints for equilibration
# ═══════════════════════════════════════════════════════════════════════════════

import os
import pathlib
import subprocess
import gmxapi as gmx

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT     = pathlib.Path(os.getcwd()).resolve()          # .../gromacs/notebooks
DATA     = ROOT.parent / 'data'                         # .../gromacs/data
WORK     = ROOT.parent / 'gmx_work'                    # .../gromacs/gmx_work
WORK.mkdir(exist_ok=True)

pdb_input = DATA / 'protein_clean.pdb'

# ── Pre-flight checks ──────────────────────────────────────────────────────────
assert pdb_input.exists(),  f"Input PDB not found : {pdb_input}"

ff_dir = pathlib.Path('/usr/local/gromacs/share/gromacs/top/charmm36-jul2022.ff')
assert ff_dir.exists(),     f"FF not found in GROMACS tree : {ff_dir}"
assert (ff_dir / 'forcefield.itp').exists(), "forcefield.itp missing — FF directory incomplete"

print(f"Input PDB  : {pdb_input}")
print(f"Work dir   : {WORK}")
print(f"Force field: {ff_dir.stem}  ({'symlink' if ff_dir.is_symlink() else 'directory'})")

# ── Run pdb2gmx ────────────────────────────────────────────────────────────────
pdb2gmx = gmx.commandline_operation(
    executable='gmx',
    arguments=[
        'pdb2gmx',
        '-ff',    'charmm36-jul2022',   # FF name without .ff suffix
        '-water', 'tip3p',              # TIP3P — consistent with CHARMM36
        '-ignh',                        # strip input H, re-add from FF database
    ],
    input_files={
        '-f': str(pdb_input),
    },
    output_files={
        '-o': str(WORK / 'processed.gro'),   # all-atom coordinates
        '-p': str(WORK / 'topol.top'),       # full topology
        '-i': str(WORK / 'posre.itp'),       # position restraints
    }
)
pdb2gmx.run()

# ── Result inspection ──────────────────────────────────────────────────────────
rc  = pdb2gmx.output.returncode.result()
err = pdb2gmx.output.stderr.result()

print(f"\nReturn code: {rc}")

if rc != 0:
    print("\n── FAILED — full stderr ────────────────────────────────────────────")
    print(err)
    raise RuntimeError("pdb2gmx failed — fix errors above before proceeding")

print("\n── Succeeded ───────────────────────────────────────────────────────────")

# 1. Force field path recorded in topol.top
#    Must point to charmm36-jul2022.ff — NOT to any /usr/local path directly
print("\n[1] FF include line in topol.top:")
for line in (WORK / 'topol.top').read_text().splitlines()[:20]:
    if '#include' in line:
        print(f"    {line}")

# 2. Output file sizes
print("\n[2] Output files:")
for fname in ['processed.gro', 'topol.top', 'posre.itp']:
    fpath = WORK / fname
    size  = fpath.stat().st_size if fpath.exists() else 0
    status = 'OK' if fpath.exists() and size > 0 else 'MISSING/EMPTY'
    print(f"    {fname:20s}  {status:8s}  {size:>10,} bytes")

# 3. Atom count — larger than input PDB because hydrogens were added
gro_lines = (WORK / 'processed.gro').read_text().splitlines()
n_atoms   = int(gro_lines[1].strip())
print(f"\n[3] Atom count in processed.gro : {n_atoms:,}")
print(f"    (input PDB had fewer — difference is added hydrogens)")

# 4. Disulfide bond detection — GROMACS reports S-S distances; review carefully
print("\n[4] Sulfur geometry (disulfide check):")
for line in err.splitlines():
    if any(kw in line for kw in ['CYS', 'MET', 'disulfide', 'SS']):
        print(f"    {line}")

# 5. Topology composition summary
print("\n[5] Topology composition:")
for line in err.splitlines():
    if any(kw in line for kw in ['pairs', 'dihedrals', 'atoms', 'residues', 'chains']):
        print(f"    {line}")

Input PDB  : /home/biplab/pys/gromacs/data/protein_clean.pdb
Work dir   : /home/biplab/pys/gromacs/gmx_work
Force field: charmm36-jul2022  (symlink)

Return code: 0

── Succeeded ───────────────────────────────────────────────────────────

[1] FF include line in topol.top:

[2] Output files:
    processed.gro         OK           209,981 bytes
    topol.top             OK         1,290,032 bytes
    posre.itp             OK            73,402 bytes

[3] Atom count in processed.gro : 4,665
    (input PDB had fewer — difference is added hydrogens)

[4] Sulfur geometry (disulfide check):
                        MET6   CYS16   MET17   CYS22   CYS38   HIS41   CYS44
       CYS16   SG117   2.145
       MET17   SD124   2.092   0.673
       CYS22   SG161   3.869   2.181   1.842
       CYS38   SG284   3.116   1.456   1.128   0.834
       CYS44   SG333   3.804   2.557   2.086   0.986   1.160   0.770
       MET49   SD370   3.772   2.818   2.240   1.398   1.564   0.900   0.653
       MET82   SD636  

In [9]:
# 1. Point to the TPR you just created
tpr_input = '../data/npt.tpr'

if os.path.exists(tpr_input):
    print(f"Loading {tpr_input} and starting MD...")
    
    # 2. Set up the simulation work handle
    # gmxapi automatically uses the resources (GPU/MPI) GROMACS was built with
    model = gmx.read_tpr(tpr_input)
    simulation = gmx.mdrun(model, runtime_args={
    '-e': 'npt_new.edr',
    '-o': 'npt_new.trr',
    '-c': 'npt_new.gro'})
    
    # 3. Run the simulation
    simulation.run()
    
    print("Simulation finished successfully!")
else:
    print(f"Could not find {tpr_input}. Make sure you are in the correct directory.")

Loading ../data/npt.tpr and starting MD...


Reading file ../data/npt.tpr, VERSION 2025.4 (single precision)
Reading file ../data/npt.tpr, VERSION 2025.4 (single precision)
Reading file /home/biplab/pys/gromacs/notebooks/mdrun_924e24ee5e1c6a65eed6c3f498b7469d_i0_0/topol.tpr, VERSION 2025.4 (single precision)


Simulation finished successfully!
